## Lesson 2

In [1]:
from ingest import load_faq_data

documents = load_faq_data()

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

139

In [4]:
documents = documents_llm

In [13]:
doc = documents[1]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

977bf7786c
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.


In [14]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [8]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [11]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [15]:
import json

user_prompt = json.dumps(doc)

In [16]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [19]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

response

ParsedResponse[TypeVar](id='resp_03531373bea03b7c006a735ce87080819d9dadd61b251e6389', created_at=1785945320.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ParsedResponseOutputMessage[TypeVar](id='msg_03531373bea03b7c006a735ce91210819da576ccf748594b9d', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{"questions":["I just found this course — is it still okay to join now, or am I too late?","Can I still enroll even if the class has already started?","If I join late, can I still get a certificate, or is that only for on-time students?","What do I need to do to be eligible for the certificate if I’m starting the course now?","Is there a deadline for project submission if I want to earn the certificate?"]}', type='output_text', logprobs=[], parsed=Questions(questions=['I just found this course — is it still okay to join now, or am I too late?', 'Can I still enroll even if the class has alre

In [18]:
result = response.output_parsed

print(result)

questions=['I just found this course late — can I still jump in now, and what do I need to do if I want the certificate?', 'If I join after the course has already started, am I still allowed to take part?', 'Is it okay to start this course anytime, or is there a deadline for joining?', 'Can latecomers still get the certificate, and is there a cutoff for the project submission?', 'I missed the course start — can I still enroll, and how does the certificate work if I do?']


In [21]:
result.questions

['I just found this course late — can I still jump in now, and what do I need to do if I want the certificate?',
 'If I join after the course has already started, am I still allowed to take part?',
 'Is it okay to start this course anytime, or is there a deadline for joining?',
 'Can latecomers still get the certificate, and is there a cutoff for the project submission?',
 'I missed the course start — can I still enroll, and how does the certificate work if I do?']

In [22]:
from evaluation_utils import llm_structured

In [24]:
result, usage = llm_structured(
    client=openai_client,
    instructions=data_gen_instructions,
    user_prompt=user_prompt,
    output_type=Questions,
)

result.questions

['I just found this course late — can I still enroll and follow along?',
 'Am I allowed to join the course even if it already started?',
 'If I sign up now, do I still get access to everything, or is it too late?',
 'Is there any deadline for joining, or can I jump into the course anytime?',
 'Can late joiners still earn a certificate, or do I have to submit the project before the submission window closes?']

In [25]:
usage.input_tokens, usage.output_tokens

(207, 103)

In [26]:
from evaluation_utils import calc_price

In [27]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.0004635,
 'total_cost': 0.0006187499999999999}

In [28]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course late — can I still enroll and follow along?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to join the course even if it already started?',
  'document': '74eb249bbf'},
 {'question': 'If I sign up now, do I still get access to everything, or is it too late?',
  'document': '74eb249bbf'},
 {'question': 'Is there any deadline for joining, or can I jump into the course anytime?',
  'document': '74eb249bbf'},
 {'question': 'Can late joiners still earn a certificate, or do I have to submit the project before the submission window closes?',
  'document': '74eb249bbf'}]